In [4]:
from google.colab import drive
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import classification_report, accuracy_score, recall_score, f1_score, roc_auc_score
import time

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
train_path = '/content/drive/MyDrive/datasets_big_data/HIGGS/train_200000.csv'
test_path = '/content/drive/MyDrive/datasets_big_data/HIGGS/validation.csv'
manifest_val_path = '/content/drive/MyDrive/datasets_big_data/HIGGS/validation.manifest.csv'

df_train = pd.read_csv(train_path, header=None)
df_test = pd.read_csv(test_path, header=None)
manifest_val = pd.read_csv(manifest_val_path)

print(f"entrenamiento: {df_train.shape}")
print(f"prueba: {df_test.shape}")
print(f"Filas en validation.csv: {len(df_test)}")
print(f"Filas en manifest_val: {len(manifest_val)}")

entrenamiento: (200000, 29)
prueba: (500000, 29)
Filas en validation.csv: 500000
Filas en manifest_val: 500000


In [6]:
X_train = df_train.drop(0, axis=1)
y_train = df_train[0]

X_test = df_test.drop(0, axis=1)
y_test = df_test[0]

nulos_train = X_train.isnull().sum().sum()
nulos_test = X_test.isnull().sum().sum()

# print(f"nulos en X_train: {nulos_train}")
# print(f"nulos en X_test: {nulos_test}")
# print(X_train.dtypes.value_counts())

# Entrenamiento


---



In [ ]:
# Busqueda de mejores hiper parámetros
param_dist = {
    'n_estimators': [100, 200, 300],          # Cantidad de arboles
    'max_depth': [10, 20, 30],          # Profundidad maxima (None = sin límite)
    'min_samples_split': [2, 5, 10],          # Muestras minimas para dividir un nodo
    'min_samples_leaf': [1, 2, 4],            # Muestras minimas en cada hoja final
    'max_features': ['sqrt', 'log2']    # Numero de características a considerar por corte
}

In [ ]:
print("Entrenando Random Forest sobre los 200,000 registros...")

# 1. El n_jobs=-1 se queda aquí (para que construya árboles en paralelo rápido)
rf_base = RandomForestClassifier(random_state=10)

# 2. Configuramos el buscador
rf_random = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist,
    n_iter=5,             # 5 combinaciones
    cv=3,                 # K-Fold de 3 pliegues
    verbose=3,            # Nivel 3 para ver el tiempo exacto de cada pliegue en consola
    random_state=10,
    scoring='roc_auc'     # Optimiza basándose en ROC-AUC
    # NOTA: Sin n_jobs aquí para evitar que Colab se congele
)

print("Iniciando búsqueda de hiperparámetros...")

# 3. Entrenamos directamente sobre tu X_train completo (200k registros)
rf_random.fit(X_train, y_train)

print("\nMejores parametros encontrados:")
print(rf_random.best_params_)
print(f"Mejor puntuación ROC-AUC (Cross-Validation): {rf_random.best_score_:.4f}")

# 4. Extraer el mejor modelo
best_rf_model = rf_random.best_estimator_

# 5. Evaluar el mejor modelo en tu conjunto de validación (X_test)
best_rf_preds = best_rf_model.predict(X_test)
best_rf_probs = best_rf_model.predict_proba(X_test)[:, 1]

print("\n--- Rendimiento del Modelo Optimizado en X_test ---")
print(f"Accuracy:  {accuracy_score(y_test, best_rf_preds):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, best_rf_probs):.4f}")

Entrenando Random Forest sobre los 200,000 registros...
Iniciando búsqueda de hiperparámetros...
Fitting 3 folds for each of 5 candidates, totalling 15 fits
[CV 1/3] END max_depth=10, max_features=log2, min_samples_leaf=1, min_samples_split=10, n_estimators=300;, score=0.786 total time= 2.7min
[CV 2/3] END max_depth=10, max_features=log2, min_samples_leaf=1, min_samples_split=10, n_estimators=300;, score=0.784 total time= 2.6min
[CV 3/3] END max_depth=10, max_features=log2, min_samples_leaf=1, min_samples_split=10, n_estimators=300;, score=0.783 total time= 2.7min
[CV 1/3] END max_depth=30, max_features=sqrt, min_samples_leaf=4, min_samples_split=10, n_estimators=300;, score=0.808 total time= 5.4min
[CV 2/3] END max_depth=30, max_features=sqrt, min_samples_leaf=4, min_samples_split=10, n_estimators=300;, score=0.807 total time= 5.5min
[CV 3/3] END max_depth=30, max_features=sqrt, min_samples_leaf=4, min_samples_split=10, n_estimators=300;, score=0.806 total time= 5.7min
[CV 1/3] END ma

In [7]:
print("\nEntrenando Extra Trees...")
et_model = ExtraTreesClassifier(n_estimators=100, random_state=10, n_jobs=-1)

start_time = time.time()
et_model.fit(X_train, y_train)
et_time = time.time() - start_time

et_preds = et_model.predict(X_test)
et_probs = et_model.predict_proba(X_test)[:, 1]


Entrenando Extra Trees...


In [13]:
print(" Entrenando RandomForest optimizado")
rf_opt = RandomForestClassifier(n_estimators=300, min_samples_split=10, min_samples_leaf=4, max_features='sqrt', max_depth=30, random_state=10)
rf_opt.fit(X_train, y_train)
rf_time = time.time() - start_time
rf_opt_preds = rf_opt.predict(X_test)
rf_opt_probs = rf_opt.predict_proba(X_test)[:, 1]

 Entrenando RandomForest optimizado


# Metricas

---



In [15]:
print("RENDIMIENTO")
mi_nombre = 'Caleb'
nombre_dataset = 'HIGGS'
tamaño_muestra = 200000
reporte_metricas = []


reporte_metricas.append({
    'member': mi_nombre,
    'dataset': nombre_dataset,
    'sample': tamaño_muestra,
    'model': 'Extra Trees Base',
    'accuracy': round(accuracy_score(y_test, et_preds), 4),
    'recall': round(recall_score(y_test, et_preds), 4),
    'f1': round(f1_score(y_test, et_preds), 4),
    'training_time_seconds': round(et_time, 2)
})

reporte_metricas.append({
    'member': mi_nombre,
    'dataset': nombre_dataset,
    'sample': tamaño_muestra,
    'model': 'Random Forest Optimizado',
    'accuracy': round(accuracy_score(y_test, rf_opt_preds), 4),
    'recall': round(recall_score(y_test, rf_opt_preds), 4),
    'f1': round(f1_score(y_test, rf_opt_preds), 4),
    'training_time_seconds': round(rf_time, 2)
})


# print(reporte_metricas)

RENDIMIENTO
[{'member': 'Caleb', 'dataset': 'HIGGS', 'sample': 200000, 'model': 'Extra Trees Base', 'accuracy': 0.7114, 'recall': 0.754, 'f1': 0.7347, 'training_time_seconds': 79.63}, {'member': 'Caleb', 'dataset': 'HIGGS', 'sample': 200000, 'model': 'Random Forest Optimizado', 'accuracy': 0.7305, 'recall': 0.758, 'f1': 0.7488, 'training_time_seconds': 1806.04}]


In [17]:

ids_limpios = manifest_val['observation_id'].str.split(':').str[1]
ids_limpios = ids_limpios.astype(int)

df_rf_final = pd.DataFrame({
    'observation_id': ids_limpios,
    'prediction': rf_opt_preds
})

df_et_final = pd.DataFrame({
    'observation_id': ids_limpios,
    'prediction': et_preds
})

# Rutas de exportación en tu Drive
rf_export_path = '/content/drive/MyDrive/datasets_big_data/HIGGS/validation_prediction.csv'
# et_export_path = '/content/drive/MyDrive/datasets_big_data/HIGGS/predicciones_et.csv'

# Exportar a CSV (index=False es vital aquí)
df_rf_final.to_csv(rf_export_path, index=False)
# df_et_final.to_csv(et_export_path, index=False)
print(f"1. {rf_export_path}")
# print(f"2. {et_export_path}")

1. /content/drive/MyDrive/datasets_big_data/HIGGS/validation_prediction.csv


In [ ]:
import json
# EXPORTACION DE PARAMETROS
mejores_parametros = rf_random.best_params_

datos_exportar = {
    "modelo": "Random Forest Optimizado",
    "roc_auc_score": round(rf_random.best_score_, 4),
    "parametros": mejores_parametros
}
ruta_config = '/content/drive/MyDrive/datasets_big_data/HIGGS/mejor_configuracion_rf.json'

with open(ruta_config, 'w') as archivo:
    json.dump(datos_exportar, archivo, indent=4)
